# Homework 5: Deploying Machine Learning Models

This notebook follows the 2026 homework and the deployment workflow from Module 5. It uses the checked-in frozen lead-scoring pipeline, serves it with the provided FastAPI application, and validates the Docker image.

In [22]:
from pathlib import Path
import hashlib
import json
import re
import shutil
import subprocess
import sys
import tempfile
import time
from urllib.error import URLError
from urllib.request import Request, urlopen


def find_homework_dir():
    current = Path.cwd().resolve()
    for directory in (current, *current.parents):
        if (directory / "homework.md").is_file() and (directory / "pipeline.bin").is_file():
            return directory
        candidate = directory / "cohorts" / "2026" / "homework" / "05-deployment"
        if (candidate / "homework.md").is_file():
            return candidate
    raise FileNotFoundError("Could not locate cohorts/2026/homework/05-deployment")


HOMEWORK_DIR = find_homework_dir()
UV = shutil.which("uv")
if UV is None:
    raise FileNotFoundError("uv was not found on PATH. Install uv, then restart the kernel.")

artifact_path = HOMEWORK_DIR / "pipeline.bin"
artifact_sha256 = hashlib.sha256(artifact_path.read_bytes()).hexdigest()
expected_sha256 = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
assert artifact_sha256 == expected_sha256, f"Unexpected model artifact checksum: {artifact_sha256}"

print(f"Homework directory: {HOMEWORK_DIR}")
print(f"Model SHA-256: {artifact_sha256}")

sync_result = subprocess.run(
    [UV, "sync", "--locked"],
    cwd=HOMEWORK_DIR,
    check=True,
    text=True,
    capture_output=True,
)
print(sync_result.stdout.strip() or "Locked environment is ready.")
if sync_result.stderr.strip():
    print(sync_result.stderr.strip())

smoke_test = subprocess.run(
    [UV, "run", "--locked", "python", "smoke_test.py"],
    cwd=HOMEWORK_DIR,
    check=True,
    text=True,
    capture_output=True,
)
print(f"Smoke test: {smoke_test.stdout.strip()}")

Homework directory: C:\Users\yanji\Projects\machine-learning-zoomcamp\cohorts\2026\homework\05-deployment
Model SHA-256: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Locked environment is ready.
Resolved 19 packages in 0.81ms
Checked 18 packages in 0.61ms
Smoke test: {'conversion_probability': 0.769799, 'conversion': True}


## Question 1: Environment Check

Record the installed `uv` version. This is a local setup check, not a graded model answer.

In [23]:
uv_version = subprocess.run(
    [UV, "--version"],
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()
print(uv_version)

uv 0.13.6 (329bf9448 2026-10-26 x86_64-pc-windows-msvc)


## Question 2: Locked Dependency

Read both project configuration files and compare the declared Scikit-Learn dependency with the resolved version in `uv.lock`.

In [24]:
import tomllib

pyproject_data = tomllib.loads((HOMEWORK_DIR / "pyproject.toml").read_text(encoding="utf-8"))
lock_data = tomllib.loads((HOMEWORK_DIR / "uv.lock").read_text(encoding="utf-8"))
declared_sklearn = next(
    dependency.split("==", 1)[1]
    for dependency in pyproject_data["project"]["dependencies"]
    if dependency.lower().startswith("scikit-learn==")
)
locked_packages = {package["name"]: package["version"] for package in lock_data["package"]}
locked_sklearn = locked_packages["scikit-learn"]
assert declared_sklearn == locked_sklearn
print(f"Scikit-Learn version: {locked_sklearn}")

Scikit-Learn version: 1.7.2


## Question 3: Load the Frozen Model

Load `pipeline.bin` with `pickle` in the locked project environment and compute the conversion probability for the supplied lead.

In [25]:
lead_for_question_3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}
prediction_script = '''
import json
import pickle
from model import normalize_record

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}
with open("pipeline.bin", "rb") as model_file:
    pipeline = pickle.load(model_file)
probability = float(pipeline.predict_proba([normalize_record(lead)])[0, 1])
print(json.dumps({"conversion_probability": probability}))
'''
question_3_run = subprocess.run(
    [UV, "run", "--locked", "python", "-c", prediction_script],
    cwd=HOMEWORK_DIR,
    check=True,
    text=True,
    capture_output=True,
)
question_3_result = json.loads(question_3_run.stdout)
question_3_probability = question_3_result["conversion_probability"]
assert 0.0 <= question_3_probability <= 1.0
print(f"Question 3 conversion probability: {question_3_probability:.3f}")

Question 3 conversion probability: 0.533


## Question 4: Serve the Model

Start the provided FastAPI application in a background process, wait for its health endpoint, and send the second lead to `/predict`. The process is stopped even if a request or assertion fails.

In [26]:
def request_json(url, payload=None):
    if payload is None:
        request = Request(url, method="GET")
    else:
        request = Request(
            url,
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )
    with urlopen(request, timeout=5) as response:
        return json.load(response)


def wait_for_health(url, process, timeout_seconds=45):
    deadline = time.monotonic() + timeout_seconds
    last_error = None
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f"Server exited with code {process.returncode}")
        try:
            return request_json(url)
        except (URLError, TimeoutError, OSError) as error:
            last_error = error
            time.sleep(0.5)
    raise TimeoutError(f"Service did not become ready: {last_error}")


API_HOST = "127.0.0.1"
API_PORT = 9696
API_BASE_URL = f"http://{API_HOST}:{API_PORT}"
project_python = subprocess.run(
    [UV, "run", "--locked", "python", "-c", "import sys; print(sys.executable)"],
    cwd=HOMEWORK_DIR,
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()
server_process = subprocess.Popen(
    [project_python, "-m", "uvicorn", "predict:app", "--host", "0.0.0.0", "--port", str(API_PORT)],
    cwd=HOMEWORK_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
try:
    local_health = wait_for_health(f"{API_BASE_URL}/health", server_process)
    assert local_health["status"] == "ok"
    assert local_health["model_sha256"] == expected_sha256

    lead_for_question_4 = {
        "lead_source": "organic_search",
        "industry": "technology",
        "employment_status": "employed",
        "location": "europe",
        "number_of_courses_viewed": 4,
        "annual_income": 80304.0,
        "interaction_count": 7,
        "lead_score": 0.74,
    }
    question_4_result = request_json(
        f"{API_BASE_URL}/predict",
        lead_for_question_4,
    )
    assert 0.0 <= question_4_result["conversion_probability"] <= 1.0
    assert isinstance(question_4_result["conversion"], bool)
    print(f"Health: {local_health}")
    print(f"Question 4 response: {question_4_result}")
    print(
        "Question 4 conversion probability: "
        f"{question_4_result['conversion_probability']:.3f}"
    )
finally:
    server_process.terminate()
    try:
        server_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait(timeout=5)

Health: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '5cb9195a7a160fdc91a1b8f57172055e33ced1994870742b937d5273f4021c67'}
Question 4 response: {'conversion_probability': 0.769799, 'conversion': True}
Question 4 conversion probability: 0.770


## Question 5: Inspect the Container Configuration

Extract the Python base-image tag declared by the first Python `FROM` instruction in the checked-in Dockerfile.

In [15]:
dockerfile_text = (HOMEWORK_DIR / "Dockerfile").read_text(encoding="utf-8")
base_image_match = re.search(r"^FROM\s+(python:[^\s@]+)", dockerfile_text, flags=re.MULTILINE)
assert base_image_match is not None, "No Python base image was found in the Dockerfile."
python_base_image = base_image_match.group(1)
assert python_base_image == "python:3.11.15-slim-bookworm"
print(f"Question 5 Python base image: {python_base_image}")

Question 5 Python base image: python:3.11.15-slim-bookworm


## Question 6: Run the Container

Build and start the checked-in image, verify `/health` and the model checksum, then run the provided `q6_test.py` client against the container. Stop the container after the checks. Docker Engine must be running.

In [30]:
import ast

docker_executable = shutil.which("docker")
if docker_executable is None:
    raise FileNotFoundError("Docker CLI is not installed.")

docker_info = subprocess.run(
    [docker_executable, "info", "--format", "{{.ServerVersion}}"],
    check=True,
    text=True,
    capture_output=True,
)
image_tag = "zoomcamp-model:2026-hw5"
subprocess.run(
    [docker_executable, "build", "-t", image_tag, "."],
    cwd=HOMEWORK_DIR,
    check=True,
    text=True,
)

container_name = f"zoomcamp-hw5-{int(time.time())}"
try:
    subprocess.run(
        [
            docker_executable,
            "run",
            "--rm",
            "-d",
            "--name",
            container_name,
            "-p",
            f"{API_PORT}:9696",
            image_tag,
        ],
        check=True,
        cwd=HOMEWORK_DIR,
        text=True,
        capture_output=True,
    )

    for _ in range(60):
        try:
            container_health = request_json(f"{API_BASE_URL}/health")
            break
        except (URLError, TimeoutError, OSError):
            time.sleep(1)
    else:
        raise TimeoutError("Container API did not become ready within 60 seconds.")

    assert container_health["status"] == "ok"
    assert container_health["model_sha256"] == expected_sha256
    print(f"Container health: {container_health}")

    q6_test = subprocess.run(
        [
            UV,
            "run",
            "--locked",
            "python",
            "q6_test.py",
            f"{API_BASE_URL}/predict",
        ],
        cwd=HOMEWORK_DIR,
        check=True,
        text=True,
        capture_output=True,
    )
    container_result = ast.literal_eval(q6_test.stdout.strip())
    assert abs(
        container_result["conversion_probability"]
        - question_4_result["conversion_probability"]
    ) <= 0.005
    print(f"q6_test.py response: {container_result}")
    print(
        "Question 6 conversion probability: "
        f"{container_result['conversion_probability']:.3f}"
    )
finally:
    subprocess.run(
        [docker_executable, "stop", container_name],
        check=False,
        text=True,
        capture_output=True,
    )

Container health: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '5cb9195a7a160fdc91a1b8f57172055e33ced1994870742b937d5273f4021c67'}
q6_test.py response: {'conversion_probability': 0.769799, 'conversion': True}
Question 6 conversion probability: 0.770


## Evaluation Summary

Collect the answers computed above. Question 6 is reported as unverified unless the container actually starts and returns a valid prediction.

In [31]:
question_6_verified = (
    docker_executable is not None
    and docker_info.returncode == 0
    and "container_result" in globals()
)
summary = {
    "uv_version": uv_version,
    "scikit_learn_version": locked_sklearn,
    "question_3_probability": f"{question_3_probability:.3f}",
    "question_4_probability": f"{question_4_result['conversion_probability']:.3f}",
    "question_5_python_base_image": python_base_image,
    "question_6_verified": question_6_verified,
}
if question_6_verified:
    summary["question_6_probability"] = f"{container_result['conversion_probability']:.3f}"
else:
    summary["question_6_status"] = "Not verified: Docker Engine was unavailable."
summary

{'uv_version': 'uv 0.13.6 (329bf9448 2026-10-26 x86_64-pc-windows-msvc)',
 'scikit_learn_version': '1.7.2',
 'question_3_probability': '0.533',
 'question_4_probability': '0.770',
 'question_5_python_base_image': 'python:3.11.15-slim-bookworm',
 'question_6_verified': True,
 'question_6_probability': '0.770'}

## Learning in Public

Review and personalize the draft below before posting. Include `#mlzoomcamp`, tag Alexey Grigorev or DataTalksClub, and remove credentials, private data, and secrets from any screenshot or recording.

In [32]:
container_post_line = (
    f"Container test: {container_result['conversion_probability']:.3f}."
    if question_6_verified
    else "Container test: pending Docker Engine verification."
)
post_draft = f'''Completed Module 5 of Machine Learning Zoomcamp 2026!

I loaded the frozen lead-scoring model, served it with FastAPI, and verified the API response.
{container_post_line}

My repo: <LINK>

#mlzoomcamp @Alexey Grigorev'''
print(post_draft)
print("\nSubmit the homework results at: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw05")

Completed Module 5 of Machine Learning Zoomcamp 2026!

I loaded the frozen lead-scoring model, served it with FastAPI, and verified the API response.
Container test: 0.770.

My repo: <LINK>

#mlzoomcamp @Alexey Grigorev

Submit the homework results at: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw05
